# Immagini + RAG + LLM senza fine-tuning

Pipeline completa:

1. Knowledge base macchine in `machine.json`.
2. Indicizzazione RAG dei manuali PDF.
3. Riconoscimento macchina da immagine con SigLIP zero-shot, senza reference image e senza fine-tuning.
4. Risposta LLM basata su macchina riconosciuta + manuali associati.


In [1]:
# ============================================================
# 0) CONFIGURAZIONE
# ============================================================

from __future__ import annotations

import base64
import json
import os
import re
import subprocess
import sys
from pathlib import Path
from typing import Dict, List

import torch
import torch.nn.functional as F
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image
from ragmens_core import RagConfig, RagIndex, ShortTermMemory, VectorMemory, ltm_text
from transformers import AutoModel, AutoProcessor

BASE_DIR = Path.cwd().resolve()

load_dotenv(BASE_DIR / ".env")

PDF_DIR = Path(os.getenv("PDF_DIR", BASE_DIR / "pdf_immagini")).resolve()
REFERENCE_IMAGES_DIR = Path(os.getenv("REFERENCE_IMAGES_DIR", BASE_DIR / "reference_images")).resolve()
MACHINE_KB_PATH = Path(os.getenv("MACHINE_KB_PATH", BASE_DIR / "machine.json")).resolve()
INDEX_DIR = Path(os.getenv("INDEX_DIR", BASE_DIR / "index_no_finetuned")).resolve()
MEM_DIR = Path(os.getenv("MEM_DIR", BASE_DIR / "memory_no_finetuned")).resolve()
OUTPUT_DEBUG_DIR = Path(os.getenv("OUTPUT_DEBUG_DIR", BASE_DIR / "outputs_debug_no_finetuned")).resolve()

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
SIGLIP_MODEL = os.getenv("SIGLIP_MODEL", "google/siglip-base-patch16-224")
LLM_BASE_URL = os.getenv("OPENAI_BASE_URL", "http://localhost:11434/v1")
LLM_MODEL = os.getenv("LLM_MODEL", "llama3.2")
OCR_BACKEND = os.getenv("OCR_BACKEND", "got")
GOT_OCR_MODEL = os.getenv("GOT_OCR_MODEL", "stepfun-ai/GOT-OCR-2.0-hf")
OCR_DEVICE = os.getenv("OCR_DEVICE", "cpu")
VISION_LLM_MODEL = os.getenv("VISION_LLM_MODEL", os.getenv("OCR_MODEL", LLM_MODEL))
VISION_LLM_BASE_URL = os.getenv("VISION_LLM_BASE_URL", LLM_BASE_URL)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "ollama")

TOP_K = int(os.getenv("TOP_K", "12"))
CONTEXT_MAX_CHARS = int(os.getenv("CONTEXT_MAX_CHARS", "16000"))
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "1200"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "220"))
MIN_CHUNK_CHARS = int(os.getenv("MIN_CHUNK_CHARS", "50"))
IMAGE_RECOGNITION_THRESHOLD = float(os.getenv("IMAGE_RECOGNITION_THRESHOLD", "0.730"))
IMAGE_RECOGNITION_MARGIN = float(os.getenv("IMAGE_RECOGNITION_MARGIN", "0.2"))
SESSION_ID = os.getenv("SESSION_ID", "image_rag_session")
FORCE_REBUILD_INDEX = os.getenv("FORCE_REBUILD_INDEX", "0") == "1"
HF_LOCAL_FILES_ONLY = os.getenv("HF_LOCAL_FILES_ONLY", "0") == "1"

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

INDEX_DIR.mkdir(parents=True, exist_ok=True)
MEM_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DEBUG_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("PDF_DIR:", PDF_DIR)
print("REFERENCE_IMAGES_DIR:", REFERENCE_IMAGES_DIR)
print("MACHINE_KB_PATH:", MACHINE_KB_PATH)
print("INDEX_DIR:", INDEX_DIR)
print("MEM_DIR:", MEM_DIR)
print("OUTPUT_DEBUG_DIR:", OUTPUT_DEBUG_DIR)
print("EMBEDDING_MODEL:", EMBEDDING_MODEL)
print("SIGLIP_MODEL:", SIGLIP_MODEL, "@", DEVICE)
print("IMAGE_RECOGNITION_THRESHOLD:", IMAGE_RECOGNITION_THRESHOLD)
print("IMAGE_RECOGNITION_MARGIN:", IMAGE_RECOGNITION_MARGIN)
print("HF_LOCAL_FILES_ONLY:", HF_LOCAL_FILES_ONLY)
print("LLM:", LLM_MODEL, "@", LLM_BASE_URL)
print("OCR_BACKEND:", OCR_BACKEND)
print("GOT_OCR_MODEL:", GOT_OCR_MODEL)
print("OCR_DEVICE:", OCR_DEVICE)
print("VISION/OCR LLM:", VISION_LLM_MODEL, "@", VISION_LLM_BASE_URL)


/Users/matteoarutadm/lavoro/digital_mens/llm_yolo/venvimmagini/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BASE_DIR: /Users/matteoarutadm/lavoro/digital_mens/llm_yolo
PDF_DIR: /Users/matteoarutadm/lavoro/digital_mens/llm_yolo/pdf_immagini
REFERENCE_IMAGES_DIR: /Users/matteoarutadm/lavoro/digital_mens/llm_yolo/reference_images
MACHINE_KB_PATH: /Users/matteoarutadm/lavoro/digital_mens/llm_yolo/machine.json
INDEX_DIR: /Users/matteoarutadm/lavoro/digital_mens/llm_yolo/index_no_finetuned
MEM_DIR: /Users/matteoarutadm/lavoro/digital_mens/llm_yolo/memory_no_finetuned
OUTPUT_DEBUG_DIR: /Users/matteoarutadm/lavoro/digital_mens/llm_yolo/outputs_debug_no_finetuned
EMBEDDING_MODEL: sentence-transformers/all-MiniLM-L6-v2
SIGLIP_MODEL: google/siglip-base-patch16-224 @ mps
IMAGE_RECOGNITION_THRESHOLD: 0.73
IMAGE_RECOGNITION_MARGIN: 0.2
HF_LOCAL_FILES_ONLY: False
LLM: llama3.2 @ http://localhost:11434/v1
OCR_BACKEND: got
GOT_OCR_MODEL: stepfun-ai/GOT-OCR-2.0-hf
OCR_DEVICE: cpu
VISION/OCR LLM: llama3.2 @ http://localhost:11434/v1


In [2]:
# ============================================================
# 1) KNOWLEDGE BASE MACCHINE
# ============================================================

DEFAULT_MACHINE_KB = [
    {
        "id": "sany_empty_container_handler",
        "macchina": "movimentatore di container vuoti sany",
        "tipo": "macchine da costruzione sany",
        "manuali": [
            "F19 - SY753757_SDCY90K7G5_Operation & Maint. Manual_IT.pdf",
        ],
    },
    {
        "id": "gru_semovente_portuale_esp10_402944",
        "macchina": "GRU SEMOVENTE PORTUALE",
        "tipo": "ESP.10 Numero di serie 402944",
        "manuali": [
            "402944_Manutenzione_ESP.10_R2.7.1-IT.pdf",
            "402944_Servizio_ESP.10_R2.7.1-IT.pdf",
        ],
    },
]

if not MACHINE_KB_PATH.exists():
    MACHINE_KB_PATH.write_text(json.dumps(DEFAULT_MACHINE_KB, ensure_ascii=False, indent=2), encoding="utf-8")

MACHINES_LIST = json.loads(MACHINE_KB_PATH.read_text(encoding="utf-8"))
MACHINES = {machine["id"]: machine for machine in MACHINES_LIST}

REFERENCE_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
REFERENCE_STOPWORDS = {"di", "da", "del", "della", "dei", "delle", "e", "ed", "il", "lo", "la", "i", "gli", "le", "un", "una", "uno", "numero", "serie", "macchina", "macchine", "tipo"}

def _normalize_for_match(text: str) -> str:
    text = (text or "").lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def _tokens_for_match(text: str) -> set[str]:
    tokens = set()
    for tok in _normalize_for_match(text).split():
        tok = re.sub(r"\d+$", "", tok)
        if len(tok) >= 3 and tok not in REFERENCE_STOPWORDS:
            tokens.add(tok)
    return tokens

def _machine_match_tokens(machine: dict) -> set[str]:
    tokens = set()
    tokens.update(_tokens_for_match(machine.get("id", "")))
    tokens.update(_tokens_for_match(machine.get("macchina", "")))
    return tokens

def _reference_match_tokens(path: Path) -> set[str]:
    return _tokens_for_match(path.stem)

def assign_reference_images(reference_dir: Path = REFERENCE_IMAGES_DIR) -> dict[str, list[str]]:
    files = [
        path for path in sorted(reference_dir.iterdir())
        if path.is_file() and path.suffix.lower() in REFERENCE_IMAGE_EXTENSIONS
    ]
    assignments: dict[str, list[str]] = {machine["id"]: [] for machine in MACHINES_LIST}
    machine_tokens = {machine["id"]: _machine_match_tokens(machine) for machine in MACHINES_LIST}

    for image_path in files:
        image_tokens = _reference_match_tokens(image_path)
        scored = []
        for machine_id, tokens in machine_tokens.items():
            overlap = len(tokens & image_tokens)
            if overlap:
                scored.append((overlap, len(tokens & image_tokens), machine_id))
        if not scored:
            continue
        scored.sort(key=lambda item: (item[0], item[1], item[2]), reverse=True)
        best_machine_id = scored[0][2]
        assignments[best_machine_id].append(str(image_path.relative_to(BASE_DIR)))

    return assignments

reference_assignments = assign_reference_images()
for machine in MACHINES_LIST:
    machine["reference_images"] = reference_assignments.get(machine["id"], [])

missing_manuals = []
for machine in MACHINES_LIST:
    for manual in machine["manuali"]:
        if not (PDF_DIR / manual).exists():
            missing_manuals.append(str(PDF_DIR / manual))

if missing_manuals:
    raise FileNotFoundError("Manuali non trovati:\n" + "\n".join(missing_manuals))

MACHINES = {machine["id"]: machine for machine in MACHINES_LIST}

print("Macchine in KB:", len(MACHINES_LIST))
for machine in MACHINES_LIST:
    print("-", machine["id"], "|", machine["macchina"], "| manuali:", len(machine["manuali"]), "| ref:", len(machine["reference_images"]))


Macchine in KB: 3
- sany_empty_container_handler | movimentatore di container vuoti sany | manuali: 1 | ref: 6
- gru_semovente_portuale_esp10_402944 | GRU SEMOVENTE PORTUALE | manuali: 2 | ref: 3
- carroponte_portuale_gen | CARROPONTE PORTUALE | manuali: 2 | ref: 4


In [3]:
# ============================================================
# 2) RAG CORE DA LIBRERIA: PDF, CHUNKING, FAISS
# ============================================================

rag_config = RagConfig(
    base_dir=BASE_DIR,
    pdf_dir=PDF_DIR,
    index_dir=INDEX_DIR,
    mem_dir=MEM_DIR,
    output_debug_dir=OUTPUT_DEBUG_DIR,
    embedding_model=EMBEDDING_MODEL,
    top_k=TOP_K,
    context_max_chars=CONTEXT_MAX_CHARS,
    force_rebuild_index=FORCE_REBUILD_INDEX,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    min_chunk_chars=MIN_CHUNK_CHARS,
)

rag = RagIndex.prepare(rag_config)

# Alias mantenuti per il resto del notebook.
chunks = rag.chunks
embedder = rag.embedder
index = rag.index
ACTIVE_CORPUS_SIGNATURE = rag.corpus_signature

print("Indice RAG pronto:", len(chunks), "chunk")


Batches: 100%|██████████| 127/127 [00:09<00:00, 13.68it/s]

Indice RAG pronto: 4034 chunk


In [4]:
# ============================================================
# 3) RETRIEVAL RAG FILTRATO PER MACCHINA
# ============================================================

def machine_document_ids(machine_id: str | None) -> List[str] | None:
    if not machine_id:
        return None
    machine = MACHINES.get(machine_id)
    if not machine:
        return []
    return list(machine.get("manuali", []))


def retrieve(query: str, machine_id: str | None = None, top_k: int = TOP_K) -> List[dict]:
    query = (query or "").strip()
    if not query:
        return []
    return rag.retrieve(
        query,
        top_k=top_k,
        document_ids=machine_document_ids(machine_id),
    )


def build_context(results: List[dict], max_chars: int = CONTEXT_MAX_CHARS) -> str:
    return rag.build_context(results, max_chars=max_chars)


def preview_retrieval(question: str, machine_id: str | None = None, top_k: int = TOP_K) -> List[dict]:
    hits = retrieve(question, machine_id=machine_id, top_k=top_k)
    for hit in hits:
        print(hit["source"], "pagina", hit["page"], "chunk", hit["chunk_index"], "score", round(hit["score"], 3))
    print("\nCONTESTO:\n")
    print(build_context(hits))
    return hits


In [5]:
# ============================================================
# 4) RICONOSCIMENTO MACCHINA CON SIGLIP SU REFERENCE IMAGES
# ============================================================

siglip_model = AutoModel.from_pretrained(SIGLIP_MODEL, local_files_only=HF_LOCAL_FILES_ONLY).to(DEVICE)
siglip_processor = AutoProcessor.from_pretrained(SIGLIP_MODEL, local_files_only=HF_LOCAL_FILES_ONLY)
siglip_model.eval()

@torch.no_grad()
def image_embedding(image_path: str | Path) -> torch.Tensor:
    image = Image.open(image_path).convert("RGB")
    inputs = siglip_processor(images=image, return_tensors="pt").to(DEVICE)
    outputs = siglip_model.get_image_features(**inputs)
    emb = outputs.pooler_output if hasattr(outputs, "pooler_output") else outputs
    emb = F.normalize(emb, dim=-1)
    return emb.squeeze(0).detach().cpu()

def build_reference_embeddings() -> dict[str, list[dict]]:
    reference_embeddings: dict[str, list[dict]] = {}
    for machine in MACHINES_LIST:
        refs = []
        for image_rel_path in machine.get("reference_images", []):
            image_path = (BASE_DIR / image_rel_path).resolve()
            if not image_path.exists():
                continue
            refs.append({
                "path": str(image_path),
                "embedding": image_embedding(image_path),
            })
        reference_embeddings[machine["id"]] = refs
    return reference_embeddings

REFERENCE_EMBEDDINGS = build_reference_embeddings()

def identify_machine(image_path: str) -> tuple[str, float, List[dict]]:
    query_emb = image_embedding(image_path)
    candidates = []

    for machine in MACHINES_LIST:
        machine_id = machine["id"]
        refs = REFERENCE_EMBEDDINGS.get(machine_id, [])
        if not refs:
            continue

        scored_refs = []
        for ref in refs:
            score = F.cosine_similarity(query_emb, ref["embedding"], dim=0).item()
            scored_refs.append((score, ref["path"]))

        best_score, best_ref_path = max(scored_refs, key=lambda item: item[0])
        candidates.append({
            "machine_id": machine_id,
            "machine": machine,
            "score": float(best_score),
            "reference_image": best_ref_path,
        })

    if not candidates:
        raise ValueError("Nessuna reference image disponibile per il riconoscimento.")

    candidates.sort(key=lambda item: item["score"], reverse=True)
    best = candidates[0]
    second = candidates[1] if len(candidates) > 1 else None
    if best["score"] < IMAGE_RECOGNITION_THRESHOLD:
        raise ValueError(
            f"Immagine non riconosciuta: score {best['score']:.3f} < {IMAGE_RECOGNITION_THRESHOLD:.2f}"
        )
    # if second and (best["score"] - second["score"] < IMAGE_RECOGNITION_MARGIN):
    #     raise ValueError(
    #         f"Immagine non riconosciuta: margine insufficiente ({best['score']:.3f} vs {second['score']:.3f}, min {IMAGE_RECOGNITION_MARGIN:.2f})"
    #     )
    return best["machine_id"], best["score"], candidates

print("Reference embeddings pronti:", sum(len(v) for v in REFERENCE_EMBEDDINGS.values()))


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Reference embeddings pronti: 13


In [6]:
# ============================================================
# 5) MEMORIA E LLM
# ============================================================

STM_MAX = 10
stm = ShortTermMemory(max_messages=STM_MAX)
ltm = VectorMemory(
    embedder=rag.embedder,
    mem_dir=rag_config.mem_dir,
    corpus_signature=rag.corpus_signature,
)


def stm_add(role: str, content: str) -> None:
    stm.add(role, content)


def stm_text(max_chars: int = 1200) -> str:
    return stm.text(max_chars=max_chars)


def memory_session_id(machine_id: str | None = None) -> str:
    suffix = machine_id or "all"
    return f"{SESSION_ID}:{suffix}"


QA_PROMPT = """Sei un assistente tecnico per macchinari industriali.
Rispondi sempre in italiano.
Usa prima le informazioni dei manuali nel CONTEXT.
Se il CONTEXT non basta, dillo in modo chiaro e suggerisci cosa verificare nel manuale o sulla macchina.
Se la macchina e riconosciuta ma seriale o modello non sono leggibili, dillo esplicitamente: puoi rispondere sull'oggetto riconosciuto, ma non puoi confermare il modello esatto dalla targhetta.
Non inventare procedure, codici errore, limiti o componenti specifici non presenti nel CONTEXT.
Cita fonti, pagine e chunk quando disponibili.

MACCHINA RICONOSCIUTA:
{machine_info}

CONFIDENZA RICONOSCIMENTO IMMAGINE:
{vision_info}

DATI LETTI DALLA FOTO / TARGHETTA:
{image_identifiers}

MEMORIA BREVE:
{stm}

MEMORIA LUNGA RILEVANTE:
{ltm}

CONTEXT DAI MANUALI:
{context}

DOMANDA:
{question}

RISPOSTA:"""


def call_llm(prompt: str) -> str:
    client = OpenAI(api_key=OPENAI_API_KEY, base_url=LLM_BASE_URL)
    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    return (response.choices[0].message.content or "").strip()


def normalize_ocr_output(result) -> str:
    if isinstance(result, str):
        return result.strip()
    if isinstance(result, list) and result:
        parts = [normalize_ocr_output(item) for item in result]
        return "\n".join(part for part in parts if part).strip()
    if isinstance(result, dict):
        for key in ["generated_text", "text", "answer"]:
            if key in result:
                return normalize_ocr_output(result[key])
        if "content" in result:
            return normalize_ocr_output(result["content"])
    return str(result).strip()


def got_ocr_text(image_path: str | Path) -> str:
    runner = BASE_DIR / "got_ocr_runner.py"
    cmd = [
        sys.executable,
        str(runner),
        str(Path(image_path).resolve()),
        "--model",
        GOT_OCR_MODEL,
        "--device",
        OCR_DEVICE,
    ]
    if HF_LOCAL_FILES_ONLY:
        cmd.append("--local-files-only")
    completed = subprocess.run(cmd, capture_output=True, text=True, check=False)
    stdout = completed.stdout.strip()
    try:
        payload = json.loads(stdout)
    except json.JSONDecodeError:
        raise RuntimeError(f"GOT-OCR output non JSON. stdout={stdout!r}; stderr={completed.stderr.strip()!r}")
    if completed.returncode != 0 or not payload.get("ok"):
        raise RuntimeError(payload.get("error") or completed.stderr.strip() or "GOT-OCR fallito")
    return str(payload.get("text") or "").strip()


def first_regex_group(patterns: list[str], text: str) -> str | None:
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.I)
        if match:
            return match.group(1).strip(" :#-_")
    return None


def parse_identifiers_from_ocr_text(text: str) -> dict:
    serial = first_regex_group([
        r"(?:serial\s*(?:no\.?|number)?|s/?n|matricola|n\.?\s*serie)\s*[:#-]?\s*([A-Z0-9][A-Z0-9./_-]{3,})",
    ], text)
    model = first_regex_group([
        r"(?:model\s*(?:no\.?|number)?|modello|type|tipo)\s*[:#-]?\s*([A-Z0-9][A-Z0-9./_-]{2,})",
    ], text)
    unclassified_code = None
    if not serial and not model:
        unclassified_code = first_regex_group([
            r"\b([A-Z]{2,}[A-Z0-9]*\d[A-Z0-9./_-]{3,})\b",
            r"\b([A-Z0-9]{4,}[-_/][A-Z0-9]{2,})\b",
        ], text.upper())
    visible_text = [line.strip() for line in text.splitlines() if line.strip()]
    return {
        "available": True,
        "manufacturer": None,
        "model_code": model,
        "serial_number": serial,
        "asset_tag": unclassified_code,
        "visible_text": visible_text,
        "raw_text": text,
        "confidence": None,
        "notes": "Estratto con GOT-OCR locale; model_code/serial_number sono stimati con regex dal testo OCR. asset_tag contiene un codice letto ma non classificato se manca un'etichetta chiara.",
    }


def identifier_summary(machine: dict, vision_score: float, image_identifiers: dict) -> dict:
    serial_number = image_identifiers.get("serial_number")
    model_code = image_identifiers.get("model_code")
    asset_tag = image_identifiers.get("asset_tag")
    ocr_available = bool(image_identifiers.get("available"))

    if model_code and serial_number:
        status = f"Oggetto riconosciuto: {machine['macchina']}. Modello letto: {model_code}. Seriale letto: {serial_number}."
        exact_model_identified = True
    elif model_code:
        status = f"Oggetto riconosciuto: {machine['macchina']}. Modello letto: {model_code}. Seriale non trovato o non leggibile."
        exact_model_identified = True
    elif serial_number:
        status = f"Oggetto riconosciuto: {machine['macchina']}. Seriale letto: {serial_number}. Modello non trovato o non leggibile."
        exact_model_identified = False
    elif asset_tag:
        status = f"Oggetto riconosciuto: {machine['macchina']}. Ho letto il codice {asset_tag}, ma non posso confermare se sia seriale o modello perche nella foto non vedo un'etichetta chiara."
        exact_model_identified = False
    elif ocr_available:
        status = f"Oggetto riconosciuto: {machine['macchina']}, ma non ho trovato il modello esatto perche non vedo un seriale o un codice modello leggibile nella foto."
        exact_model_identified = False
    else:
        status = f"Oggetto riconosciuto: {machine['macchina']}, ma non posso confermare il modello esatto perche l'OCR non e disponibile o ha fallito."
        exact_model_identified = False

    return {
        "object_name": machine["macchina"],
        "machine_type": machine.get("tipo"),
        "vision_score": vision_score,
        "model_code": model_code,
        "serial_number": serial_number,
        "asset_tag": asset_tag,
        "exact_model_identified": exact_model_identified,
        "status": status,
    }


def image_data_url(image_path: str | Path) -> str:
    path = Path(image_path)
    mime = "image/png" if path.suffix.lower() == ".png" else "image/jpeg"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{encoded}"


def extract_json_object(text: str) -> dict:
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.S)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                pass
    return {"raw_response": text}


def extract_image_identifiers(image_path: str | Path, machine: dict | None = None) -> dict:
    """Legge targhetta, seriale e codice modello dalla foto con OCR locale o modello vision."""
    if OCR_BACKEND == "got":
        try:
            return parse_identifiers_from_ocr_text(got_ocr_text(image_path))
        except Exception as exc:
            return {"available": False, "backend": "got", "error": str(exc)}

    machine_hint = ""
    if machine:
        machine_hint = json.dumps({k: machine.get(k) for k in ["id", "macchina", "tipo", "manuali"]}, ensure_ascii=False)

    prompt = f"""Leggi la foto come OCR tecnico industriale.
Devi estrarre solo testo visibile: targhetta, serial number, model number, part number, marca, codici macchina.
Non inventare nulla. Se un dato non e leggibile, usa null.
Contesto macchina possibile: {machine_hint}

Rispondi solo con JSON valido in questo schema:
{{
  "manufacturer": string|null,
  "model_code": string|null,
  "serial_number": string|null,
  "asset_tag": string|null,
  "visible_text": [string],
  "confidence": number,
  "notes": string|null
}}"""

    client = OpenAI(api_key=OPENAI_API_KEY, base_url=VISION_LLM_BASE_URL)
    try:
        response = client.chat.completions.create(
            model=VISION_LLM_MODEL,
            temperature=0,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": image_data_url(image_path)}},
                ],
            }],
        )
    except Exception as exc:
        return {"available": False, "error": str(exc)}

    parsed = extract_json_object(response.choices[0].message.content or "")
    parsed["available"] = True
    return parsed


def build_prompt(question: str, context: str, machine: dict, vision_candidates: List[dict], memory_hits: List[dict], image_identifiers: dict | None = None) -> str:
    machine_info = json.dumps({k: machine[k] for k in ["id", "macchina", "tipo", "manuali"]}, ensure_ascii=False, indent=2)
    vision_info = "\n".join(
        f"- {item['machine']['macchina']} ({item['machine_id']}): {item['score']:.3f}; reference: {item['reference_image']}"
        for item in vision_candidates
    )
    return QA_PROMPT.format(
        machine_info=machine_info,
        vision_info=vision_info,
        image_identifiers=json.dumps(image_identifiers or {}, ensure_ascii=False, indent=2),
        stm=stm_text(),
        ltm=ltm_text(memory_hits),
        context=context,
        question=question,
    )


def ask_machine(image_path: str, question: str, top_k: int = TOP_K, save_memory: bool = True) -> Dict[str, object]:
    try:
        machine_id, vision_score, vision_candidates = identify_machine(image_path)
    except ValueError as exc:
        print(str(exc))
        return {
            "image_path": image_path,
            "question": question,
            "recognized": False,
            "reason": str(exc),
        }
    machine = MACHINES[machine_id]

    image_identifiers = extract_image_identifiers(image_path, machine)
    identifier_text = " ".join(
        str(value)
        for key, value in image_identifiers.items()
        if key not in {"available", "error", "notes"} and value
    )
    recognition_summary = identifier_summary(machine, vision_score, image_identifiers)
    rag_query = f"{machine['macchina']} {machine['tipo']} {identifier_text} {question}"
    hits = retrieve(rag_query, machine_id=machine_id, top_k=top_k)
    context = build_context(hits)
    memory_hits = ltm.search(memory_session_id(machine_id), question, top_k=4)
    prompt = build_prompt(question, context, machine, vision_candidates, memory_hits, image_identifiers)
    answer = call_llm(prompt)

    stm_add("user", f"[{machine['macchina']}] {question}")
    stm_add("assistant", answer)
    if save_memory:
        ltm.add_turn(memory_session_id(machine_id), question, answer)

    print("\n=== RIEPILOGO RICONOSCIMENTO ===")
    print(recognition_summary["status"])
    print(f"Confidenza riconoscimento oggetto: {vision_score:.3f}")
    print(f"\n=== OGGETTO RICONOSCIUTO ===\n{machine['macchina']} | {machine['tipo']}")
    print("\n=== DATI LETTI DALLA FOTO / TARGHETTA ===")
    print(json.dumps(image_identifiers, ensure_ascii=False, indent=2))
    print("\n=== RISPOSTA ===\n")
    print(answer)
    print("\n=== FONTI RECUPERATE ===")
    for hit in hits:
        print(f"- {hit['source']} | pagina {hit['page']} | chunk {hit['chunk_index']} | score {hit['score']:.3f}")

    return {
        "image_path": image_path,
        "question": question,
        "machine_id": machine_id,
        "machine": machine,
        "vision_score": vision_score,
        "vision_candidates": vision_candidates,
        "image_identifiers": image_identifiers,
        "recognition_summary": recognition_summary,
        "answer": answer,
        "hits": hits,
        "context": context,
        "prompt": prompt,
    }


def ask_known_machine(machine_id: str, question: str, top_k: int = TOP_K, save_memory: bool = True) -> Dict[str, object]:
    if machine_id not in MACHINES:
        raise ValueError(f"machine_id non valido: {machine_id}. Valori: {list(MACHINES)}")
    machine = MACHINES[machine_id]
    hits = retrieve(f"{machine['macchina']} {machine['tipo']} {question}", machine_id=machine_id, top_k=top_k)
    context = build_context(hits)
    memory_hits = ltm.search(memory_session_id(machine_id), question, top_k=4)
    fake_candidates = [{"machine_id": machine_id, "machine": machine, "score": 1.0, "best_prompt": "macchina scelta manualmente"}]
    prompt = build_prompt(question, context, machine, fake_candidates, memory_hits)
    answer = call_llm(prompt)
    if save_memory:
        ltm.add_turn(memory_session_id(machine_id), question, answer)
    print(answer)
    return {"question": question, "machine_id": machine_id, "machine": machine, "answer": answer, "hits": hits, "context": context, "prompt": prompt}

print("Pipeline pronta.")


Pipeline pronta.


In [7]:
# ============================================================
# 6) ESEMPI DI USO
# ============================================================

# Caso completo: immagine -> SigLIP riconosce oggetto -> OCR legge seriale/modello -> RAG sui manuali -> LLM
risultato = ask_machine(
    image_path="reference_images/movimentatore_di_container6.jpeg",
    question="Riconosci l'oggetto, leggi seriale o modello se visibili, e dimmi su quale modello/manuale devo lavorare. e dimmi cosa posso verificare per controllare se la pompa idraulica funziona correttamente?",
)

# # Caso senza immagine, utile per testare solo RAG + LLM su una macchina nota:
# risultato = ask_known_machine(
#     machine_id="gru_semovente_portuale_esp10_402944",
#     question="Come verifico il corretto funzionamento della pompa idraulica?",
# )

# # Anteprima retrieval senza chiamare LLM:
# preview_retrieval(
#     "Come verifico il corretto funzionamento della pompa idraulica?",
#     machine_id="gru_semovente_portuale_esp10_402944",
# )


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



=== RIEPILOGO RICONOSCIMENTO ===
Oggetto riconosciuto: movimentatore di container vuoti sany, ma non posso confermare il modello esatto perche l'OCR non e disponibile o ha fallito.
Confidenza riconoscimento oggetto: 1.000

=== OGGETTO RICONOSCIUTO ===
movimentatore di container vuoti sany | macchine da costruzione sany

=== DATI LETTI DALLA FOTO / TARGHETTA ===
{
  "available": false,
  "backend": "got",
  "error": "GotOcr2ForConditionalGeneration.__init__() got an unexpected keyword argument 'dtype'"
}

=== RISPOSTA ===

Grazie per avermi fornito le informazioni necessarie!

Dopo aver analizzato il contesto e i manuali forniti, posso concludere che l'oggetto riconosciuto è effettivamente un movimentatore di container vuoti SANY.

Tuttavia, non sono in grado di confermare il modello esatto della macchina a causa della mancanza di informazioni sulla targhetta o sulle serie del modello.

Per quanto riguarda la pompa idraulica, posso suggerire alcuni passaggi per controllare se funziona 